# 04 — Monte Carlo

**Workload:** Seeded Monte Carlo, bootstrap confidence intervals, `default_rng`, and `RandomState`.

This notebook is executed against the RNP engine. Every output below is
stored in the notebook and visible when rendered on GitHub.

In [1]:
from pathlib import Path
import importlib.util
import sys

PROJECT_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "shim" / "rnp_numpy").is_dir()
)
for path in (PROJECT_ROOT / "harness" / "_redirect", PROJECT_ROOT / "shim"):
    sys.path.insert(0, str(path))

# IPython may preload the oracle NumPy, so clear that namespace before
# executing the exact redirect hook used by examples/run_all.py.
for module_name in list(sys.modules):
    if module_name == "numpy" or module_name.startswith("numpy."):
        del sys.modules[module_name]
redirect_path = PROJECT_ROOT / "harness" / "_redirect" / "sitecustomize.py"
redirect_spec = importlib.util.spec_from_file_location("_rnp_notebook_redirect", redirect_path)
redirect = importlib.util.module_from_spec(redirect_spec)
redirect_spec.loader.exec_module(redirect)
import numpy as np

probe = np.array(0)
print("numpy version:", np.__version__)
print(f"RNP engine active: {np.__name__} ({type(probe).__module__}.{type(probe).__name__})")
assert np.__name__ == "rnp_numpy"

numpy version: 2.5.2
RNP engine active: rnp_numpy (_rnp.ndarray)


## Estimate pi

Sample points in a square and count those inside the unit circle.

In [2]:
rng = np.random.default_rng(8675309)
points = rng.uniform(-1.0, 1.0, size=(20_000, 2))
inside_count = np.count_nonzero(np.sum(points * points, axis=1) <= 1.0)
pi_estimate = 4.0 * inside_count / points.shape[0]
print("inside circle / 20000:", inside_count)
print("pi estimate:", pi_estimate)

inside circle / 20000: 15831
pi estimate: 3.1662


## Bootstrap a confidence interval

Resample a seeded observation vector and take percentile bounds.

In [3]:
observations = rng.normal(loc=12.0, scale=2.5, size=80)
bootstrap_indices = rng.integers(0, observations.size, size=(1_000, observations.size))
bootstrap_means = observations[bootstrap_indices].mean(axis=1)
bootstrap_ci = np.percentile(bootstrap_means, [2.5, 97.5])
print("bootstrap 95% CI:", np.round(bootstrap_ci, 6))

bootstrap 95% CI: [11.790463 12.663034]


## Exercise the legacy RNG

The playbook covers both the modern generator and `RandomState`.

In [4]:
legacy = np.random.RandomState(1969)
legacy_rolls = legacy.randint(1, 7, size=24)
print("legacy first rolls:", legacy_rolls[:8])

legacy first rolls: [2 4 5 1 6 2 4 5]


## Verify the result

In [5]:
assert np.array_equal(inside_count, 15831)
assert np.allclose(pi_estimate, 3.1662, rtol=0.0, atol=0.0)
assert np.array_equal(legacy_rolls[:8], [2, 4, 5, 1, 6, 2, 4, 5])
print("PASS — all Monte Carlo assertions passed.")

PASS — all Monte Carlo assertions passed.
